# NB_CursorUsage_Bronze_To_Silver
**Source:** `AI_Medallion.Bronze.Cursor_CursorUsage`  
**Targets (reads):** `OktaUserforAI`, `HC_Silver_Historical`  
**Target:** `AI_Medallion.Silver.CursorUsage`  
**Run after:** NB_OktaUserforAI_Bronze_To_Silver, NB_HCHistorical_Bronze_To_Silver

In [ ]:
%run ./common/common_bootstrap


In [9]:
b
r
o
n
z
e
_
d
f
 
=
 
s
p
a
r
k
.
t
a
b
l
e
(
T
A
B
L
E
_
C
U
R
S
O
R
_
B
R
O
N
Z
E
)


i
f
 
C
U
R
S
O
R
_
B
R
O
N
Z
E
_
F
I
L
E
N
A
M
E
:


 
 
 
 
b
r
o
n
z
e
_
d
f
 
=
 
b
r
o
n
z
e
_
d
f
.
f
i
l
t
e
r
(
F
.
c
o
l
(
"
f
i
l
e
n
a
m
e
"
)
 
=
=
 
C
U
R
S
O
R
_
B
R
O
N
Z
E
_
F
I
L
E
N
A
M
E
)


b
r
o
n
z
e
 
=
 
b
r
o
n
z
e
_
d
f




o
k
t
a
 
=
 
s
p
a
r
k
.
t
a
b
l
e
(
T
A
B
L
E
_
O
K
T
A
_
S
I
L
V
E
R
)


h
c
 
=
 
s
p
a
r
k
.
t
a
b
l
e
(
T
A
B
L
E
_
H
C
_
S
I
L
V
E
R
)




u
s
e
r
_
n
o
r
m
 
=
 
F
.
l
o
w
e
r
(
F
.
t
r
i
m
(
F
.
c
o
l
(
"
U
s
e
r
"
)
)
)




StatementMeta(, bb9bb06d-c4e0-40aa-b603-8e642e55f61c, 23, Finished, Available, Finished, False)

In [10]:
base = (
    bronze.filter(is_valid_email(F.col("User")))
    .select(
        F.col("Date").cast("date").alias("CursorUsageDate"),
        user_norm.alias("CursorUsageUser"),
        F.col("`Service Account Name`").alias("CursorUsageServiceAccountName"),
        F.col("`Service Account ID`").alias("CursorUsageServiceAccountID"),
        F.col("`Cloud Agent ID`").alias("CursorUsageCloudAgentID"),
        F.col("`Automation ID`").alias("CursorUsageAutomationID"),
        F.col("Kind").alias("CursorUsageKind"),
        F.col("Model").alias("CursorUsageModel"),
        F.col("`Max Mode`").alias("CursorUsageMaxMode"),
        F.col("`Input (w/ Cache Write)`").cast("long").alias("CursorUsageInputWithCacheWrite"),
        F.col("`Input (w/o Cache Write)`").cast("long").alias("CursorUsageInputWithoutCacheWrite"),
        F.col("`Cache Read`").cast("long").alias("CursorUsageCacheRead"),
        F.col("`Output Tokens`").cast("long").alias("CursorUsageOutputTokens"),
        F.col("`Total Tokens`").cast("long").alias("CursorUsageTotalTokens"),
        F.col("Cost").cast("decimal(18,8)").alias("CursorUsageCost"),
        F.col("FileName").alias("CursorUsageFileName"),
        F.col("ExecutionDate").cast("timestamp").alias("CursorUsageIngestionDate"),
    )
    .withColumn(
        "CursorUsageMonthNo",
        (F.year("CursorUsageDate") * F.lit(100) + F.month("CursorUsageDate")).cast("int"),
    )
)

okta_slim = okta.select(
    "OktaUserId",
    "OktaUserEmail",
    "OktaUserFullEmail",
    "OktaUserEmployeeNumber",
    "OktaUserFullName",
    "OktaUserRegion",
    "OktaUserCountry",
)

# Inner join: match login or full email
joined = base.join(
    okta_slim,
    (F.col("CursorUsageUser") == F.col("OktaUserEmail"))
    | (F.col("CursorUsageUser") == F.col("OktaUserFullEmail")),
    "inner",
)

joined = joined.withColumn(
    "CursorUsageKeyEmail",
    F.when(
        F.col("OktaUserEmail").isNotNull(),
        F.concat(
            F.col("CursorUsageMonthNo").cast("string"),
            F.lit("/"),
            F.col("OktaUserEmail"),
        ),
    ),
).withColumn(
    "CursorUsageKeyFullEmail",
    F.when(
        F.col("OktaUserFullEmail").isNotNull(),
        F.concat(
            F.col("CursorUsageMonthNo").cast("string"),
            F.lit("/"),
            F.col("OktaUserFullEmail"),
        ),
    ),
)

hc_slim = hc.select(
    F.col("HCKeyEmail"),
    F.col("HCBusinessUnit"),
    F.col("HCBusinessUnitCode"),
    F.col("HCDivision"),
    F.col("HCJobTitle"),
    F.col("HCOffice"),
    F.col("HCTalentSegment"),
    F.col("Month_No").alias("HCMonthNo"),
    F.col("OktaUserCountry").alias("HCOktaUserCountry"),
    F.col("HCDivisionINC"),
)

joined = joined.join(
    hc_slim,
    (F.col("HCKeyEmail") == F.col("CursorUsageKeyEmail"))
    | (F.col("HCKeyEmail") == F.col("CursorUsageKeyFullEmail")),
    "left",
).withColumn(
    "HCDivisionINC",
    F.coalesce(F.col("HCDivisionINC"), F.lit(HC_DIVISION_INC_DEFAULT)),
)

write_df= joined.distinct()

write_df=write_df.withColumn(
    FACT_CURSOR_ACTIVE_MERGE_KEY,
    cursor_usage_record_key()
    )

StatementMeta(, bb9bb06d-c4e0-40aa-b603-8e642e55f61c, 24, Finished, Available, Finished, False)

In [11]:
merge_incremental(
    spark,
    write_df,
    TABLE_CURSOR_SILVER,
    FACT_CURSOR_ACTIVE_MERGE_KEY,
)

StatementMeta(, bb9bb06d-c4e0-40aa-b603-8e642e55f61c, 25, Finished, Cancelled, Cancelled, False)

Py4JJavaError: An error occurred while calling o5704.saveAsTable.
: org.apache.spark.SparkException: Job 29 cancelled part of cancelled job group 25
	at org.apache.spark.scheduler.DAGScheduler.failJobAndIndependentStages(DAGScheduler.scala:3241)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:3111)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleJobGroupCancelled$4(DAGScheduler.scala:1278)
	at scala.runtime.java8.JFunction1$mcVI$sp.apply(JFunction1$mcVI$sp.java:23)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:79)
	at org.apache.spark.scheduler.DAGScheduler.handleJobGroupCancelled(DAGScheduler.scala:1277)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3403)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive(DAGScheduler.scala:3379)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive(DAGScheduler.scala:3368)
	at org.apache.spark.util.EventLoop$$an